# Extremum-on-the-Residual — Batch Processing of All `Data_raw` Scans

**Method** (from `Baseline_Method_Comparison.ipynb`, §2e). Each BPD scope trace is
an 81.25 MHz laser pulse train (period $T \approx 12.31$ ns). The 5th laser pulse
(~52 ns) is the one electro-optically modulated by the THz field; the 4th pulse
(~40 ns), one period earlier, is the clean unmodulated reference.

1. **Period-fill baseline removal.** Tile the clean pre-window period (the 4th
   peak) forward across the signal window $[45,100]$ ns to build a baseline that
   tracks the unmodulated laser carrier, whether the EO pulse ends up above or
   below it. `residual = trace - baseline`.
2. **Extremum on the residual.** Instead of reading the residual's mean at one
   *fixed* probe time (which misses the EO feature when it shifts in time with
   delay), slide the same ±25-sample averaging window across a search range
   around the peak-5 region and take the **signed value at the position of
   largest fluctuation**. Smoothing before extracting (a sliding boxcar, not a
   raw single-sample `argmax`) keeps the noise floor low while still tracking the
   moving peak — the same idea as `extremas.py`, applied to the residual instead
   of the raw trace.

This notebook applies that exact method to **every compatible `.npz` scan** under
`Data_raw/` (the 54 files that carry `BPDwf` + `phase` + `x` — i.e. every
phase/delay scan of the balanced photodiode). For each file it produces:

- **one plot** of the recovered signed EO voltage (mV) vs. laser-beam delay (ps)
  for that scan, and
- **one CSV** with `phase_deg`, `delay_ps`, and `voltage_mV` for every delay step.

Outputs are written to `Results/ExtremumResidual/<source folder>/`, mirroring
`Data_raw`'s own folder layout (`Aug01/`, `July31_/`, `July30_/`, and `root/` for
files directly under `Data_raw/`).

`fiberlaser/` is excluded — those files are single-shot timing/jitter captures
(real epoch timestamps, no phase scan, no repeating BPD peak structure), not
compatible with the peak4/peak5 residual method.

## 0 · Setup — constants, discovery, output dirs

In [1]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.ndimage import uniform_filter1d
import csv

# Run from repo root so Data_raw/ resolves whether launched from Analysis/ or root
if os.path.basename(os.getcwd()) == 'Analysis':
    os.chdir('..')

DATA_ROOT = 'Data_raw'
OUT_ROOT  = 'Results/ExtremumResidual'
os.makedirs(OUT_ROOT, exist_ok=True)

# instrument / analysis constants (mirror Baseline_Method_Comparison.ipynb)
RF_HZ           = 81.25e6          # 81.25 MHz phase reference
T_LASER_NS      = 1e9 / RF_HZ      # 12.31 ns laser period
MV              = 1e3              # V -> mV
SIGNAL_START_NS = 45.0
SIGNAL_END_NS   = 100.0
PROBE_HALF_WIN  = 25                # +/- samples for the residual averaging window
LINE_COLOR      = '#3168B0'         # single consistent hue for every trace plot
print(f'Laser period T = {T_LASER_NS:.3f} ns')

Laser period T = 12.308 ns


In [2]:
# Excluded: fiberlaser/ (different measurement type -- see markdown above)
EXCLUDE_DIRS = {'fiberlaser'}

def source_folder(path):
    '''Folder name directly under Data_raw/, or "root" for files at the top level.'''
    rel = os.path.relpath(path, DATA_ROOT)
    parts = rel.split(os.sep)
    return parts[0] if len(parts) > 1 else 'root'

all_npz = sorted(glob.glob(os.path.join(DATA_ROOT, '**', '*.npz'), recursive=True))
grouped = {}
for f in all_npz:
    folder = source_folder(f)
    if folder in EXCLUDE_DIRS:
        continue
    grouped.setdefault(folder, []).append(f)

for folder in sorted(grouped):
    grouped[folder].sort(key=lambda p: os.path.basename(p).lower())

n_total = sum(len(v) for v in grouped.values())
print(f'{n_total} candidate .npz scan files across {len(grouped)} folders:')
for folder in sorted(grouped):
    print(f'  {folder:>8}: {len(grouped[folder])} files')

54 candidate .npz scan files across 4 folders:
     Aug01: 13 files
   July30_: 2 files
   July31_: 38 files
      root: 1 files


## 1 · The method — period-fill baseline + sliding-window residual extremum

In [3]:
def delay_ps_from_phase(phase_deg):
    return (phase_deg - phase_deg[0]) / RF_HZ / 360 * 1e12


def period_fill_indices(t_ns, s0, s1, T_ns):
    """For every sample in [s0,s1] return the index one clean laser period before it."""
    dt = (t_ns[-1] - t_ns[0]) / (len(t_ns) - 1)
    T  = T_ns / dt
    i0, i1 = int(round(s0 / dt)), int(round(s1 / dt))
    idx = np.arange(i0, i1)
    nl  = np.ceil((idx + 1 - i0) / T).astype(int)
    return i0, i1, np.round(idx - nl * T).astype(int)


def process_scan(path):
    """Apply period-fill baseline removal + sliding-window residual extremum to one
    Data_raw scan file. Returns (result, None) or (None, reason) if incompatible."""
    d = np.load(path, allow_pickle=True)
    if not {'BPDwf', 'phase', 'x'}.issubset(d.files):
        return None, 'missing BPDwf/phase/x'
    bpd, x, phase = d['BPDwf'], d['x'], d['phase']
    if bpd.size == 0 or phase.size == 0:
        return None, 'empty arrays'

    t_ns = 1e9 * x[0, :]
    dt = t_ns[1] - t_ns[0]
    traces = bpd.mean(axis=1)                      # (steps, samples) -- shot-averaged
    delay_ps = delay_ps_from_phase(phase)
    n_steps, n_samp = traces.shape

    # locate the laser peaks on this file's own grand-mean trace
    gm = traces.mean(axis=0)
    peak_idxs, _ = find_peaks(gm, distance=int(0.8 * T_LASER_NS / dt))
    if len(peak_idxs) < 5:
        return None, f'only {len(peak_idxs)} laser peaks found (need >=5)'
    pk5 = peak_idxs[4]

    # period-fill baseline: tile the clean period one T earlier across the signal window
    s1 = min(SIGNAL_END_NS, t_ns[-1])
    i0, i1, sl = period_fill_indices(t_ns, SIGNAL_START_NS, s1, T_LASER_NS)
    sl = np.clip(sl, 0, n_samp - 1)
    baseline = traces.copy()
    baseline[:, i0:i1] = traces[:, sl]
    residual = traces - baseline                    # signed, full detector bandwidth

    # probe time: sample of largest variance across delay (needs >=5 steps to be meaningful)
    tp = int(residual.var(axis=0).argmax()) if n_steps >= 5 else int(pk5)

    # slide the +/-PROBE_HALF_WIN averaging window across a +/-0.3T search range,
    # take the signed value at the position of largest fluctuation (extremum on residual)
    win_width = min(2 * PROBE_HALF_WIN + 1, n_samp)
    smoothed = uniform_filter1d(residual, win_width, axis=1, mode='nearest')
    sh = int(0.3 * T_LASER_NS / dt)
    sw = slice(max(0, tp - sh), min(n_samp, tp + sh))
    seg = smoothed[:, sw]
    j = np.argmax(np.abs(seg), axis=1)
    voltage_V = np.take_along_axis(seg, j[:, None], axis=1)[:, 0]

    return {
        'phase_deg': phase,
        'delay_ps': delay_ps,
        'voltage_V': voltage_V,
        'n_steps': n_steps,
        'tp_ns': t_ns[tp],
    }, None

## 2 · Batch run — process every scan, save a plot + CSV per file, sorted by folder

In [4]:
def plot_trace(folder, stem, result, out_png):
    delay_ps = result['delay_ps']
    v_mv = MV * result['voltage_V']
    fig, ax = plt.subplots(figsize=(9, 4.5))
    if result['n_steps'] > 1:
        ax.plot(delay_ps, v_mv, color=LINE_COLOR, lw=1.6)
    else:
        ax.plot(delay_ps, v_mv, 'o', color=LINE_COLOR, ms=8)
    ax.axhline(0, color='0.4', lw=0.8, ls='--', alpha=0.7)
    ax.set_xlabel('laser-beam delay (ps)')
    ax.set_ylabel('signed EO voltage, extremum on residual (mV)')
    ax.set_title(f'{folder}/{stem}  ({result["n_steps"]} steps, probe @ {result["tp_ns"]:.2f} ns)',
                 fontsize=10)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(out_png, dpi=170, bbox_inches='tight')
    plt.close(fig)


def save_csv(result, out_csv):
    with open(out_csv, 'w', newline='') as fh:
        w = csv.writer(fh)
        w.writerow(['phase_deg', 'delay_ps', 'voltage_mV'])
        for p, dps, v in zip(result['phase_deg'], result['delay_ps'], MV * result['voltage_V']):
            w.writerow([f'{p:.6f}', f'{dps:.6f}', f'{v:.6f}'])


summary = []      # (folder, stem, n_steps, vmin_mV, vmax_mV)
skipped = []      # (folder, stem, reason)

for folder in sorted(grouped):
    out_dir = os.path.join(OUT_ROOT, folder)
    os.makedirs(out_dir, exist_ok=True)
    for path in grouped[folder]:
        stem = os.path.splitext(os.path.basename(path))[0]
        result, reason = process_scan(path)
        if result is None:
            skipped.append((folder, stem, reason))
            continue
        out_png = os.path.join(out_dir, stem + '.png')
        out_csv = os.path.join(out_dir, stem + '.csv')
        plot_trace(folder, stem, result, out_png)
        save_csv(result, out_csv)
        v_mv = MV * result['voltage_V']
        summary.append((folder, stem, result['n_steps'], v_mv.min(), v_mv.max()))

print(f'processed {len(summary)} files, skipped {len(skipped)}')

processed 53 files, skipped 1


## 3 · Summary

In [5]:
print(f'{"folder":>8}  {"file":<55} {"steps":>6} {"min mV":>9} {"max mV":>9}')
for folder, stem, n_steps, vmin, vmax in summary:
    print(f'{folder:>8}  {stem:<55} {n_steps:>6} {vmin:>9.1f} {vmax:>9.1f}')

if skipped:
    print(f'\nskipped ({len(skipped)}):')
    for folder, stem, reason in skipped:
        print(f'  {folder}/{stem}: {reason}')

print(f'\nAll plots + CSVs written under {OUT_ROOT}/<folder>/, one pair per scan file.')

  folder  file                                                     steps    min mV    max mV
   Aug01  fine scan1nCNoBBO-BBO-DipoleOn_Shifted                     228     -25.2      18.8
   Aug01  fine scan1nCNoBBO-NoSD-DipoleOff_Shifted                   228     -64.3      62.5
   Aug01  fine scan1nCNoBBO-NoSD-WithIrisDipoleOff_rotated45Shifted    228    -209.5     211.5
   Aug01  fine scan1nCNoBBO-NoSD-WithIrisDipoleOff_rotated90Shifted    228    -127.4     186.4
   Aug01  fine scan1nCNoBBO-NoSD-WithIrisDipoleOff_Shifted           228    -190.6     170.5
   Aug01  FineScan1nC                                                228     -23.4      30.6
   Aug01  FineScan1nCDipoleOn                                        228      -8.5       8.0
   Aug01  FineScan1nCNoBBODipoleOff                                  228     -62.4      57.1
   Aug01  FineScan1nCNoBBODipoleOff_Shifted                          228     -62.1      59.0
   Aug01  FineScan1nCOneBBODipoleOff_Shifted                      